# SMS backup cleanup (runs inside *your own* Google account)

Each person runs this notebook while signed in to the Google account that owns the backups.
Nothing is downloaded to your computer and nothing leaves your Google account: Colab runs in Google's cloud and reads your Drive directly.

**Steps:** 1) mount Drive → 2) set the folder → 3) `analyze` (read-only, re-run until finished) → 4) read the report → 5) `apply` → 6) delete the `superseded` folder and **empty the Drive trash**.

Tip: *Runtime → Change runtime type* is not needed; the free CPU runtime is fine. If Colab disconnects, just run the cells again — progress is saved on Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Get the tool. If the repo is private, use a fine-grained read-only token:
#   !git clone https://<TOKEN>@github.com/LegitUnicorn/SMS-Cleanup.git
!rm -rf /content/SMS-Cleanup && git clone -q https://github.com/LegitUnicorn/SMS-Cleanup.git /content/SMS-Cleanup
%cd /content/SMS-Cleanup

In [ ]:
# ---- EDIT THIS: the Drive folder that holds sms-YYYYMMDDHHMMSS.xml / calls-*.xml ----
# Either a path under /content/drive/MyDrive, or just paste the folder's link from drive.google.com
BACKUP_DIR = '/content/drive/MyDrive/SMSBackupRestore'

if BACKUP_DIR.startswith('http'):
    from smscleanup.colab import resolve_folder_link
    BACKUP_DIR = resolve_folder_link(BACKUP_DIR)  # also checks you own the folder
WORK_DIR   = BACKUP_DIR.rstrip('/') + '-cleanup'   # state, report, recovered + superseded files

import os, re, collections
PAT = re.compile(r'^(sms|calls)-\d{14}.*\.xml$', re.I)

def find_backup_folders(root='/content/drive/MyDrive'):
    hits = collections.Counter()
    for d, subdirs, files in os.walk(root):
        subdirs[:] = [s for s in subdirs if not s.endswith('-cleanup')]
        n = sum(1 for f in files if PAT.match(f))
        if n:
            hits[d] = n
    return hits

files = sorted(f for f in os.listdir(BACKUP_DIR) if PAT.match(f)) if os.path.isdir(BACKUP_DIR) else []
if not files:
    print(f'No sms-/calls- backup files in {BACKUP_DIR}')
    print('Searching your whole Drive for them (can take a minute)...')
    hits = find_backup_folders()
    for d, n in hits.most_common():
        print(f'  {n:5d} backup files in {d}')
    if hits:
        print('\n>>> Set BACKUP_DIR above to the folder with the most files and re-run this cell.')
    else:
        print('Nothing found under My Drive. If the backups are in "Shared with me", '
              'add a shortcut to that folder in My Drive first.')
else:
    total = sum(os.path.getsize(os.path.join(BACKUP_DIR, f)) for f in files)
    print(f'{len(files)} backup files, {total/1e9:.1f} GB in {BACKUP_DIR}')
    for f in files[:5] + (['...'] if len(files) > 10 else []) + files[-5:]:
        print('  ', f)

In [ ]:
# Optional sanity check on the newest backup (counts + date range)
sms_files = sorted(f for f in files if f.lower().startswith('sms-'))
if sms_files:
    newest = os.path.join(BACKUP_DIR, sms_files[-1])
    !python -m smscleanup inspect "$newest"
else:
    print('No sms-*.xml files found yet -- fix BACKUP_DIR in the cell above first.')

## Analyze (read-only for your backup folder)
Reads every backup once, newest → oldest. Safe to stop and re-run at any time; it continues where it left off.
`--max-hours 10` keeps it inside a Colab session; just run it again afterwards.

In [ ]:
!python -m smscleanup analyze "$BACKUP_DIR" --work-dir "$WORK_DIR" --max-hours 10 --tmp-dir /content/tmp

In [ ]:
from IPython.display import Markdown
Markdown(open(os.path.join(WORK_DIR, 'report.md')).read())

### Only if the report shows read errors
Google Drive sometimes returns `Input/output error` in the middle of a big file. Those files are kept (never removed) until they can be read completely. This re-reads **only** those files, then re-run the report cell above.

In [ ]:
!python -m smscleanup analyze "$BACKUP_DIR" --work-dir "$WORK_DIR" --retry-errors --tmp-dir /content/tmp

## Apply
Before touching anything this re-verifies, from disk, that every message in every file it is about to remove is present in a kept file or in `recovered/`.
Superseded files are **moved** to `WORK_DIR/superseded` (not deleted) so you can look before you delete.

In [ ]:
!python -m smscleanup apply "$BACKUP_DIR" --work-dir "$WORK_DIR" --yes

## Free the space
1. In drive.google.com open the `…-cleanup/superseded` folder, select all, delete.
2. Open **Trash → Empty trash**. Storage is only released after this (it can take a few hours for the quota meter to update).
3. **Keep** `…-cleanup/recovered/*.xml` — those hold messages that were in old backups only (e.g. deleted from the phone later). They are valid backup files you can restore with the app.